# SEC 台积电季度指引 2011–2026 原始提交清点

冻结 219 个官方 6-K 元数据候选、64 个季度月份，按接收顺序逐一读取至首个匹配的收入前瞻；原始字节备份 NAS。

In [1]:
import json,hashlib,subprocess,time,tarfile,tempfile,shlex,re
from pathlib import Path
from datetime import datetime,timezone
from collections import defaultdict
from bs4 import BeautifulSoup
from src.data.nas_market_data import _settings,nas_api_call
root=Path.cwd();card=root/'research/stage1_tsmc_sec_6k_guidance_source_probe_20261023';raw=root/'data/snapshots/stage1_tsmc_sec_6k_guidance_source_probe_20261023';plan=json.loads((card/'quarterly_inventory_plan.json').read_text())
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and not (card/'quarterly_inventory_state.json').exists()
for name,key in [('result.json','body_sha256'),('acceptance_clock_result.json','acceptance_sha256'),('submissions_index_result.json','submissions_sha256')]:assert sha(card/name)==plan['parent_source_results'][key]
for name,h in plan['input_raw_sha256'].items():assert sha(raw/name)==h
sources=[]
for name in ('recent','older'):
 d=json.loads((raw/f'submissions_{name}.json').read_text());s=d['filings']['recent'] if name=='recent' else d
 for i,form in enumerate(s['form']):
  day=s['filingDate'][i]
  if form=='6-K' and '2011'<=day[:4]<='2026' and day[5:7] in ('01','04','07','10') and 11<=int(day[8:10])<=31:
   sources.append({'month':day[:7],'filed':day,'accepted_api':s['acceptanceDateTime'][i],'accession':s['accessionNumber'][i]})
assert len(sources)==plan['candidate_accession_count_expected']==219 and len({x['month'] for x in sources})==plan['quarter_month_count_expected']==64 and len({x['accession'] for x in sources})==219
by_month=defaultdict(list)
for row in sources:by_month[row['month']].append(row)
for month in by_month:by_month[month].sort(key=lambda x:(x['accepted_api'],x['accession']))
state={'status':'started','GET_attempts':0,'months_completed':0,'model_fits':0,'native_accounts':0,'market_outcome_reads':0};(card/'quarterly_inventory_state.json').write_text(json.dumps(state,indent=2)+'\n');receipts=[];matches=[];unknown_months=[];failure=None
ordinal={1:'fourth',4:'first',7:'second',10:'third'}
for month in sorted(by_month):
 expected=ordinal[int(month[5:7])];match=None
 for candidate in by_month[month]:
  acc=candidate['accession'];reused=acc in plan['reuse_full_submission_accessions']
  if reused:body=raw/plan['reuse_full_submission_accessions'][acc];assert sha(body)==plan['input_raw_sha256'][body.name]
  else:
   body=raw/f'quarterly_{acc}.txt';headers=raw/f'quarterly_{acc}.headers';assert not body.exists();state['GET_attempts']+=1;assert state['GET_attempts']<=plan['request']['GETs_max'];(card/'quarterly_inventory_state.json').write_text(json.dumps(state,indent=2)+'\n');started=datetime.now(timezone.utc).isoformat()
   url=plan['exact_request_template'].format(accession_no_hyphen=acc.replace('-',''),accession=acc)
   cmd=['curl','--http1.1','--noproxy','*','--connect-timeout','10','--max-time','60','--max-filesize','8000000','--fail-with-body','-sS','-L','-D',str(headers),'-o',str(body),'-A',plan['request']['user_agent'],'-w','%{http_code} %{url_effective}',url]
   try:
    run=subprocess.run(cmd,capture_output=True,text=True,timeout=75);parts=run.stdout.rsplit(' ',1);status=int(parts[0]) if len(parts)==2 and parts[0].isdigit() else None
    receipt={'month':month,'accession':acc,'url':url,'requested_at_utc':started,'received_at_utc':datetime.now(timezone.utc).isoformat(),'curl_exit_code':run.returncode,'http_status':status,'final_url':parts[-1] if parts else None,'stderr':run.stderr[:250]}
   except Exception as exc:receipt={'month':month,'accession':acc,'url':url,'requested_at_utc':started,'received_at_utc':datetime.now(timezone.utc).isoformat(),'exception':str(exc)[:250]}
   receipt.update(raw_sha256=sha(body) if body.exists() else None,raw_bytes=body.stat().st_size if body.exists() else 0,headers_sha256=sha(headers) if headers.exists() else None)
   receipts.append(receipt);(card/'quarterly_inventory_receipts.json').write_text(json.dumps(receipts,indent=2)+'\n')
   if receipt.get('curl_exit_code')!=0 or receipt.get('http_status')!=200 or not body.exists():failure=f'{month}_{acc}_transport';break
   time.sleep(plan['request']['seconds_between_requests'])
  try:
   b=body.read_bytes();head=b[:12000].decode('utf-8',errors='replace');assert '<SEC-DOCUMENT>' in head.upper() and acc in head and re.search(r'CENTRAL INDEX KEY:\s*0*1046179',head)
   accepted=re.search(r'ACCEPTANCE-DATETIME>\s*(\d{14})',head);assert accepted
   parts=[]
   for doc in re.findall(rb'<DOCUMENT>(.*?)</DOCUMENT>',b,re.S|re.I):
    typ=re.search(rb'<TYPE>\s*([^\r\n<]+)',doc,re.I);content=re.search(rb'<TEXT>(.*?)</TEXT>',doc,re.S|re.I)
    if typ and content and typ.group(1).strip().upper() in (b'6-K',b'EX-99.1'):
     parts.append(BeautifulSoup(content.group(1).decode('utf-8',errors='replace'),'html.parser').get_text(' ',strip=True))
   text=' '.join(parts)
   title=re.search(r'TSMC\s+Reports\s+(First|Second|Third|Fourth)\s+Quarter\s+EPS',text,re.I)
   wanted=title and title.group(1).lower()==expected
   forward=re.search(r'Revenue\s+(?:is\s+expected\s+to\s+be|to\s+be)\s+between\s+.{0,150}',text,re.I)
   nums=re.findall(r'\d+(?:,\d{3})*(?:\.\d+)?',forward.group(0)) if forward else []
   if wanted and forward and len(nums)>=2:
    low=float(nums[0].replace(',',''));high=float(nums[1].replace(',',''));assert low<high,(month,low,high)
    match={'reporting_month':month,'reported_quarter_ordinal':expected,'accession':acc,'filed':candidate['filed'],'accepted_header':accepted.group(1),'accepted_api':candidate['accepted_api'],'raw_sha256':sha(body),'raw_name':body.name,'reused_anchor':reused,'forward_revenue_range_excerpt':forward.group(0)[:180],'first_two_numeric_values':[low,high]}
    break
  except Exception as exc:failure=f'{month}_{acc}_identity_{type(exc).__name__}_{str(exc)[:100]}';break
 if failure:break
 if match:matches.append(match)
 else:unknown_months.append(month)
 state['months_completed']+=1;(card/'quarterly_inventory_state.json').write_text(json.dumps(state,indent=2)+'\n');(card/'quarterly_inventory_progress.json').write_text(json.dumps({'matched':matches,'unknown_months':unknown_months,'GET_attempts':state['GET_attempts']},indent=2)+'\n')
files=[{'name':x.name,'sha256':sha(x),'bytes':x.stat().st_size} for x in sorted(raw.iterdir()) if x.name.startswith('quarterly_')];manifest={'plan_sha256':sha(card/'quarterly_inventory_plan.json'),'files':files,'receipts_sha256':sha(card/'quarterly_inventory_receipts.json') if (card/'quarterly_inventory_receipts.json').exists() else None,'progress_sha256':sha(card/'quarterly_inventory_progress.json') if (card/'quarterly_inventory_progress.json').exists() else None};(card/'quarterly_inventory_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
settings=_settings(root)
with tempfile.TemporaryDirectory(prefix='tsmc-sec-quarterly-') as td:
 archive=Path(td)/'source.tar.gz'
 with tarfile.open(archive,'w:gz') as t:
  for x in files:t.add(raw/x['name'],arcname='raw/'+x['name'])
  for name in ('quarterly_inventory_plan.json','quarterly_inventory_manifest.json'):
   t.add(card/name,arcname=name)
  for name in ('quarterly_inventory_receipts.json','quarterly_inventory_progress.json'):
   if (card/name).exists():t.add(card/name,arcname=name)
 digest=sha(archive);a=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact'];assert a['sha256']==digest and a['size_bytes']==archive.stat().st_size
 remote=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest
 with tempfile.TemporaryDirectory(prefix='tsmc-sec-quarterly-restore-') as rd:
  restored=Path(rd)/'restore.tar.gz'
  with restored.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(remote)],stdout=out,stderr=subprocess.PIPE,timeout=300)
  assert run.returncode==0 and sha(restored)==digest
  with tarfile.open(restored,'r:gz') as t:
   for x in files:
    m=t.extractfile('raw/'+x['name']);assert m and hashlib.sha256(m.read()).hexdigest()==x['sha256']
result={'status':'SEC_TSMC_quarterly_guidance_2011_2026_all_64_months_identified' if not failure and len(matches)==64 else 'bounded_SEC_TSMC_quarterly_inventory_partial_or_failed','failure':failure,'plan_sha256':sha(card/'quarterly_inventory_plan.json'),'GET_attempts':state['GET_attempts'],'candidate_accessions':219,'quarter_months':64,'matched_months':len(matches),'unknown_months':unknown_months,'matches':matches,'NAS_archive_sha256':digest,'NAS_remote_and_scoped_restore_verified':True,'all_first_public_issuer_clocks_qualified':False,'numeric_semantics_qualified':False,'market_outcome_reads':0,'model_fits':0,'native_accounts':0};(card/'quarterly_inventory_result.json').write_text(json.dumps(result,indent=2)+'\n');state.update(status='completed' if result['status'].endswith('identified') else 'partial_or_failed',NAS_archive_sha256=digest);(card/'quarterly_inventory_state.json').write_text(json.dumps(state,indent=2)+'\n');print({'status':result['status'],'failure':failure,'GETs':state['GET_attempts'],'matched_months':len(matches),'unknown_months':len(unknown_months),'NAS':digest})


{'status': 'SEC_TSMC_quarterly_guidance_2011_2026_all_64_months_identified', 'failure': None, 'GETs': 82, 'matched_months': 64, 'unknown_months': 0, 'NAS': '9d6eb73637bb84f10408c9e42d1d89eda0e30a0c3e8e9fd8969876329291e016'}


## 中文结论

2011–2026 64/64 季度月份找到首份含下一季收入指引的 SEC 原始提交；82 次新 GET，NAS 归档恢复通过。 详见 `conclusion.md`。